In [0]:
from pyspark.sql.functions import (
    col,
    trim,
    current_timestamp
)

print("Silver transformation environment ready!")

Silver transformation environment ready!


In [0]:
# Source and destination tables
bronze_tables = [
    "customers",
    "geolocation",
    "order_items",
    "order_payments",
    "order_reviews",
    "orders",
    "products",
    "sellers",
    "category_translation"
]

for table_name in bronze_tables:

    print(f"\nProcessing: {table_name}")

    # Read Bronze Delta table
    df = spark.table(f"workspace.bronze.{table_name}")

    bronze_count = df.count()

    # Exclude ingestion metadata from business transformations
    business_columns = [
        c for c in df.columns
        if c not in ["_ingested_at", "_source_file"]
    ]

    # Remove completely empty business records
    df = df.dropna(
        how="all",
        subset=business_columns
    )

    # Trim whitespace from string columns
    for field in df.schema.fields:
        if field.dataType.simpleString() == "string":
            df = df.withColumn(
                field.name,
                trim(col(field.name))
            )

    # Remove duplicate business records
    df = df.dropDuplicates(business_columns)

    # Add Silver processing timestamp
    silver_df = df.withColumn(
        "_silver_processed_at",
        current_timestamp()
    )

    # Save as Silver Delta table
    (
        silver_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"workspace.silver.{table_name}")
    )

    silver_count = silver_df.count()

    print(f"Bronze records: {bronze_count}")
    print(f"Silver records: {silver_count}")
    print(f"Records removed: {bronze_count - silver_count}")

print("\nSILVER TRANSFORMATION COMPLETED!")


Processing: customers
Bronze records: 99441
Silver records: 99441
Records removed: 0

Processing: geolocation
Bronze records: 1000163
Silver records: 738332
Records removed: 261831

Processing: order_items
Bronze records: 112650
Silver records: 112650
Records removed: 0

Processing: order_payments
Bronze records: 103886
Silver records: 103886
Records removed: 0

Processing: order_reviews
Bronze records: 99224
Silver records: 99224
Records removed: 0

Processing: orders
Bronze records: 99441
Silver records: 99441
Records removed: 0

Processing: products
Bronze records: 32951
Silver records: 32951
Records removed: 0

Processing: sellers
Bronze records: 3095
Silver records: 3095
Records removed: 0

Processing: category_translation
Bronze records: 71
Silver records: 71
Records removed: 0

SILVER TRANSFORMATION COMPLETED!
